In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("AnalisisRegistroCSV")
    .getOrCreate()
)

print("Versión de Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/09 16:15:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Versión de Spark: 3.5.1


In [2]:
#Load

In [3]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv("/opt/spark-data/sales.csv")
)

In [4]:
df.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(c)
    for c in df.columns
]).show(truncate=False)

+---------+------------+-------------+------+---------+----------+------------+
|order_id | order_date | customer_id | city | product | quantity | unit_price |
+---------+------------+-------------+------+---------+----------+------------+
|0        |0           |0            |0     |0        |0         |0           |
+---------+------------+-------------+------+---------+----------+------------+



In [5]:
df.printSchema()

root
 |-- order_id : double (nullable = true)
 |--  order_date : timestamp (nullable = true)
 |--  customer_id : string (nullable = true)
 |--  city : string (nullable = true)
 |--  product : string (nullable = true)
 |--  quantity : double (nullable = true)
 |--  unit_price : double (nullable = true)



In [6]:
df.show()

+---------+-------------------+-------------+--------------+-----------+----------+------------+
|order_id |        order_date | customer_id |         city |   product | quantity | unit_price |
+---------+-------------------+-------------+--------------+-----------+----------+------------+
|   1001.0|2026-09-01 00:00:00|        C001 | Barranquilla |   Teclado |       2.0|     85000.0|
|   1002.0|2026-09-01 00:00:00|        C002 |       Bogotá |     Mouse |       3.0|     45000.0|
|   1003.0|2026-09-02 00:00:00|        C003 |     Medellín |   Monitor |       1.0|    780000.0|
|   1004.0|2026-09-02 00:00:00|        C001 | Barranquilla |     Mouse |       2.0|     45000.0|
|   1005.0|2026-09-03 00:00:00|        C004 |         Cali |   Teclado |       1.0|     85000.0|
|   1006.0|2026-09-03 00:00:00|        C005 |       Bogotá |   Monitor |       2.0|    780000.0|
|   1007.0|2026-09-04 00:00:00|        C006 |     Medellín | Audifonos |       2.0|    120000.0|
|   1008.0|2026-09-04 00:00:00

In [7]:
#Transfor

In [8]:
df_cleaned = df.toDF(*[c.strip() for c in df.columns])

In [9]:
def clean_text(column):
    return F.lower(
        F.translate(
            F.regexp_replace(F.col(column), " ", ""),
            "áéíóúÁÉÍÓÚäëïöüÄËÏÖÜñÑ",
            "aeiouAEIOUaeiouAEIOUnN"
        )
    )

columns_to_clean = ["city", "product"]

for col in columns_to_clean:
    df_cleaned = df_cleaned.withColumn(col, clean_text(col))

In [10]:
df_valid = df_cleaned.filter(F.col("quantity") > 0) \
                 .withColumn("quantity", F.abs(F.col("quantity"))) \
                 .withColumn("unit_price", F.abs(F.col("unit_price")))

df_invalid = df_cleaned.filter(F.col("quantity") <= 0) \
                 .withColumn("quantity", F.abs(F.col("quantity"))) \
                 .withColumn("unit_price", F.abs(F.col("unit_price")))

In [11]:
df_valid = df_valid.withColumn("sale_total", F.col("quantity") * F.col("unit_price"))

#Report

In [15]:
df_valid.select(F.count("*").alias("register_total")).show()
df_invalid.select(F.count("*").alias("register_total")).show()
df_valid.show()

+--------------+
|register_total|
+--------------+
|            13|
+--------------+

+--------------+
|register_total|
+--------------+
|             2|
+--------------+

+--------+-------------------+-----------+------------+---------+--------+----------+----------+
|order_id|         order_date|customer_id|        city|  product|quantity|unit_price|sale_total|
+--------+-------------------+-----------+------------+---------+--------+----------+----------+
|  1001.0|2026-09-01 00:00:00|      C001 |barranquilla|  teclado|     2.0|   85000.0|  170000.0|
|  1002.0|2026-09-01 00:00:00|      C002 |      bogota|    mouse|     3.0|   45000.0|  135000.0|
|  1003.0|2026-09-02 00:00:00|      C003 |    medellin|  monitor|     1.0|  780000.0|  780000.0|
|  1004.0|2026-09-02 00:00:00|      C001 |barranquilla|    mouse|     2.0|   45000.0|   90000.0|
|  1005.0|2026-09-03 00:00:00|      C004 |        cali|  teclado|     1.0|   85000.0|   85000.0|
|  1006.0|2026-09-03 00:00:00|      C005 |      bogo

In [19]:
df_valid.select(
    F.round(F.mean("sale_total"),2).alias("mean_sale"),
    F.min("sale_total").alias("sale_min"),
    F.max("sale_total").alias("sale_max")
).show()

+---------+--------+---------+
|mean_sale|sale_min| sale_max|
+---------+--------+---------+
|365769.23| 85000.0|1560000.0|
+---------+--------+---------+



In [ ]:
df_valid.select(
    F.round(F.mean("sale_total"),2).alias("mean_sale"),
    F.min("sale_total").alias("sale_min"),
    F.max("sale_total").alias("sale_max")
).show()

In [18]:
df_mean_city = df_valid.groupBy("city").agg(
    F.round(F.mean("sale_total"),2).alias("mean_sale"),
    F.min("sale_total").alias("sale_min"),
    F.max("sale_total").alias("sale_max")
)

df_mean_city.show()

+------------+---------+--------+---------+
|        city|mean_sale|sale_min| sale_max|
+------------+---------+--------+---------+
|        cali| 432500.0| 85000.0| 780000.0|
|    medellin| 387500.0|170000.0| 780000.0|
|barranquilla|146666.67| 90000.0| 180000.0|
|      bogota| 475000.0| 85000.0|1560000.0|
+------------+---------+--------+---------+

